# Results & Model Comparison

This notebook aggregates and presents results from training all 5 algorithms with hyperparameter tuning.

**Prerequisites**: Run `python scripts/train_pipeline.py` first to generate trained models and metrics.

## 1. Setup and Load Results

In [ ]:
import sys
from pathlib import Path
import json

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

In [ ]:
# Load results
results_dir = Path.cwd().parent / 'results'
models_dir = Path.cwd().parent / 'models'

# Load metrics
metrics_file = results_dir / 'metrics.json'
if metrics_file.exists():
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    print("✅ Metrics loaded successfully!")
    print(f"Models evaluated: {[k for k in metrics.keys() if k != 'hyperparameters']}")
else:
    print("❌ metrics.json not found. Please run: python scripts/train_pipeline.py")
    metrics = None

In [ ]:
# Load comparison table
comparison_file = results_dir / 'performance_comparison.csv'
if comparison_file.exists():
    comparison_df = pd.read_csv(comparison_file)
    print("\nPerformance Comparison Table:")
    print(comparison_df.to_string(index=False))
else:
    print("❌ performance_comparison.csv not found")
    comparison_df = None

## 2. Performance Summary

In [ ]:
if metrics:
    # Extract model metrics (excluding hyperparameters)
    model_metrics = {k: v for k, v in metrics.items() if k != 'hyperparameters'}
    
    # Find best model by macro F1
    best_model = max(model_metrics.items(), key=lambda x: x[1].get('f1_macro', 0))
    best_name, best_scores = best_model
    
    print("="*80)
    print("PERFORMANCE SUMMARY")
    print("="*80)
    print(f"\nBest Model: {best_name.replace('_', ' ').upper()}")
    print(f"  Accuracy:      {best_scores['accuracy']:.4f}")
    print(f"  Macro F1:      {best_scores['f1_macro']:.4f}")
    print(f"  Weighted F1:   {best_scores['f1_weighted']:.4f}")
    print(f"  Precision:     {best_scores['precision_macro']:.4f}")
    print(f"  Recall:        {best_scores['recall_macro']:.4f}")
    
    # Baseline comparison
    baseline = model_metrics.get('baseline', {})
    if baseline:
        improvement = ((best_scores['f1_macro'] - baseline['f1_macro']) / baseline['f1_macro'] * 100)
        print(f"\nBaseline (Stratified Random):")
        print(f"  Macro F1:      {baseline['f1_macro']:.4f}")
        print(f"\nImprovement over baseline: {improvement:.1f}%")

## 3. Model Ranking by Macro F1

In [ ]:
if metrics:
    # Prepare data for ranking
    model_metrics_clean = {k: v for k, v in metrics.items() if k != 'hyperparameters'}
    
    ranking_data = []
    for model_name, scores in model_metrics_clean.items():
        ranking_data.append({
            'Model': model_name.replace('_', ' ').title(),
            'Macro F1': scores.get('f1_macro', 0),
            'Accuracy': scores.get('accuracy', 0)
        })
    
    ranking_df = pd.DataFrame(ranking_data).sort_values('Macro F1', ascending=False)
    
    print("\nModel Ranking by Macro F1:")
    for idx, row in ranking_df.iterrows():
        print(f"{idx+1}. {row['Model']:30s} - F1: {row['Macro F1']:.4f}, Acc: {row['Accuracy']:.4f}")

In [ ]:
# Visualization: Model ranking
if metrics:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bars = ax.barh(ranking_df['Model'], ranking_df['Macro F1'], color='steelblue')
    ax.set_xlabel('Macro F1 Score', fontsize=12, fontweight='bold')
    ax.set_title('Model Ranking by Macro F1 Score', fontsize=14, fontweight='bold')
    ax.set_xlim([0, 1])
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2,
                f'{width:.4f}', ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 4. Detailed Metrics Comparison

In [ ]:
if metrics:
    # Create detailed comparison table
    detailed_data = []
    for model_name, scores in model_metrics_clean.items():
        detailed_data.append({
            'Algorithm': model_name.replace('_', ' ').title(),
            'Accuracy': f"{scores.get('accuracy', 0):.4f}",
            'Precision': f"{scores.get('precision_macro', 0):.4f}",
            'Recall': f"{scores.get('recall_macro', 0):.4f}",
            'Macro F1': f"{scores.get('f1_macro', 0):.4f}",
            'Weighted F1': f"{scores.get('f1_weighted', 0):.4f}"
        })
    
    detailed_df = pd.DataFrame(detailed_data)
    print("\nDetailed Performance Metrics:")
    print(detailed_df.to_string(index=False))

## 5. Metrics Comparison Visualization

In [ ]:
if metrics:
    # Prepare data for multi-metric comparison
    metrics_list = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']
    models_list = [k for k in model_metrics_clean.keys()]
    
    # Create comparison matrix
    comparison_matrix = []
    for metric in metrics_list:
        metric_values = []
        for model in models_list:
            value = model_metrics_clean[model].get(metric, 0)
            metric_values.append(value)
        comparison_matrix.append(metric_values)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(models_list))
    width = 0.15
    
    colors = ['steelblue', 'coral', 'lightgreen', 'gold', 'plum']
    
    for idx, metric in enumerate(metrics_list):
        offset = (idx - 2) * width
        ax.bar(x + offset, comparison_matrix[idx], width, label=metric.replace('_', ' ').title(), color=colors[idx])
    
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Multi-Metric Comparison Across Algorithms', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ').title() for m in models_list], rotation=45, ha='right')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Best Model vs Baseline

In [ ]:
if metrics and 'baseline' in model_metrics_clean:
    best_model_name = best_name.replace('_', ' ').title()
    best_scores = model_metrics_clean[best_name]
    baseline_scores = model_metrics_clean['baseline']
    
    # Data for comparison
    metrics_to_compare = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']
    best_values = [best_scores[m] for m in metrics_to_compare]
    baseline_values = [baseline_scores[m] for m in metrics_to_compare]
    metric_labels = [m.replace('_', ' ').title() for m in metrics_to_compare]
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(metric_labels))
    width = 0.35
    
    ax.bar(x - width/2, best_values, width, label=f'Best Model: {best_model_name}', color='steelblue')
    ax.bar(x + width/2, baseline_values, width, label='Baseline (Stratified Random)', color='coral')
    
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Best Model vs Baseline Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, rotation=45, ha='right')
    ax.legend(fontsize=11)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Hyperparameter Tuning Summary

In [ ]:
if metrics and 'hyperparameters' in metrics:
    print("Hyperparameter Tuning Results:")
    print("="*80)
    
    hyperparams = metrics['hyperparameters']
    for model_name, hp_info in hyperparams.items():
        print(f"\n{model_name.replace('_', ' ').upper()}")
        print(f"  Best CV Score: {hp_info['best_score']:.4f}")
        print(f"  Best Parameters: {hp_info['best_params']}")

## 8. Key Findings & Interpretation

In [ ]:
if metrics:
    print("="*80)
    print("KEY FINDINGS & INTERPRETATION")
    print("="*80)
    
    best_name_readable = best_name.replace('_', ' ').upper()
    best_f1 = best_scores['f1_macro']
    best_acc = best_scores['accuracy']
    baseline_f1 = model_metrics_clean['baseline']['f1_macro']
    improvement = ((best_f1 - baseline_f1) / baseline_f1 * 100)
    
    print(f"\n1. BEST PERFORMING ALGORITHM")
    print(f"   {best_name_readable}")
    print(f"   Test-set Macro F1: {best_f1:.4f}")
    print(f"   Test-set Accuracy: {best_acc:.4f}")
    print(f"   Improvement over baseline: {improvement:.1f}%")
    
    print(f"\n2. ALGORITHM COMPARISON")
    print(f"   Worst performing: {ranking_df.iloc[-1]['Model']} (F1: {ranking_df.iloc[-1]['Macro F1']:.4f})")
    print(f"   Score spread: {best_f1 - ranking_df.iloc[-1]['Macro F1']:.4f}")
    
    print(f"\n3. BASELINE PERFORMANCE")
    print(f"   Baseline Macro F1: {baseline_f1:.4f}")
    print(f"   Baseline Accuracy: {model_metrics_clean['baseline']['accuracy']:.4f}")
    print(f"   Status: {'✅ Model significantly outperforms' if improvement > 10 else '⚠️ Model marginally outperforms' if improvement > 0 else '❌ Model underperforms'} baseline")
    
    print(f"\n4. CLINICAL IMPLICATIONS")
    if best_acc > 0.95:
        print(f"   ✅ High accuracy ({best_acc:.4f}) - potentially suitable for clinical deployment with validation")
    elif best_acc > 0.80:
        print(f"   ⚠️ Moderate accuracy ({best_acc:.4f}) - suitable for decision support but not standalone diagnosis")
    else:
        print(f"   ❌ Low accuracy ({best_acc:.4f}) - requires further refinement before clinical use")
    
    print(f"\n5. NEXT STEPS")
    print(f"   1. Review demo.ipynb for inference examples")
    print(f"   2. Analyze per-class performance for problematic classes")
    print(f"   3. Consider feature engineering or selection if performance is suboptimal")
    print(f"   4. External validation on independent test set recommended")
    print(f"\n" + "="*80)